# Clase 3 · 05 · Vectorial: clientes que se comportan parecido

Una base vectorial (Pinecone, pgvector, Mosaic AI Vector Search) guarda **vectores** y responde *"¿cuáles son los k más parecidos a este?"*. Los vectores pueden venir de un modelo de IA (embeddings de texto o imágenes) o construirse a mano.

Acá cada cliente se representa con un vector de **10 números** que resume su comportamiento:

| Dimensiones | Significado |
|---|---|
| 5 | Proporción del gasto en cada categoría (`electronics`, `home`, `books`, `sports`, `fashion`) |
| 3 | Proporción de pagos por canal (`card`, `wallet`, `transfer`) |
| 2 | Ticket promedio (dividido por 2.000) y proporción de compras de más de 1.500 |

Pregunta: *dado un cliente marcado, ¿qué otros clientes se le parecen?*

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)

customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
for table in [customers, products, transactions]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá 00_preflight (y la clase 2)."
lab_path = f"{config.volume_path}/nosql_lab"
print(f"Esquema: {config.catalog}.{config.schema} · escala {config.scale}")

## 1. Construir los vectores

Agregamos con Spark (una fila por cliente) y recién después pasamos a NumPy: son pocos miles de filas.

In [ ]:
categories = ["electronics", "home", "books", "sports", "fashion"]
channels = ["card", "wallet", "transfer"]
vectors_table = qualified_table(config, "nosql_customer_vectors")

base = (spark.table(transactions).join(spark.table(products).select("product_id", "category"), "product_id")
        .withColumn("amount", F.col("amount").cast("double")))
spend = base.groupBy("customer_id").pivot("category", categories).agg(F.sum("amount"))
pays = (base.groupBy("customer_id").pivot("payment_channel", channels).agg(F.count("transaction_id"))
        .select("customer_id", *[F.col(c).alias(f"n_{c}") for c in channels]))
stats = base.groupBy("customer_id").agg(F.sum("amount").alias("total"), F.count("*").alias("n"),
                                        F.avg("amount").alias("avg_amount"), F.avg((F.col("amount") > 1500).cast("double")).alias("high_ticket_share"),
                                        F.sum("is_fraud").alias("fraud_transactions"))
features = stats.join(spend, "customer_id").join(pays, "customer_id").na.fill(0).orderBy("customer_id").toPandas()

X = np.column_stack(
    [features[c] / features["total"] for c in categories]
    + [features[f"n_{c}"] / features["n"] for c in channels]
    + [features["avg_amount"] / 2000, features["high_ticket_share"]]
).astype("float32")
ids = features["customer_id"].to_numpy()
flagged = (features["fraud_transactions"] >= 3).to_numpy()      # mismo criterio que en 04_grafos
dims = categories + channels + ["avg_amount", "high_ticket_share"]
print(f"Matriz de vectores: {X.shape[0]:,} clientes × {X.shape[1]} dimensiones · marcados: {flagged.mean():.1%}")
display(pd.DataFrame(X[:5], columns=dims, index=ids[:5]).round(3))

Guardamos los vectores como una tabla `id + vector + metadatos`, que es exactamente lo que se carga en una base vectorial.

In [ ]:
vector_rows = pd.DataFrame({"customer_id": ids.astype("int64"), "embedding": [v.tolist() for v in X], "flagged": flagged})
spark.createDataFrame(vector_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(vectors_table)
display(spark.table(vectors_table).limit(3))

## 2. Ver los vectores en 2D con PCA

No podemos dibujar 10 dimensiones. **PCA** encuentra las dos direcciones con mayor varianza y proyecta cada cliente sobre ellas.

In [ ]:
centered = X - X.mean(axis=0)
_, singular_values, components = np.linalg.svd(centered, full_matrices=False)
explained = singular_values ** 2 / (singular_values ** 2).sum()
xy = centered @ components[:2].T

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [3, 2]})
axes[0].scatter(xy[~flagged, 0], xy[~flagged, 1], s=6, alpha=0.35, color="#94a3b8", label="no marcado")
axes[0].scatter(xy[flagged, 0], xy[flagged, 1], s=10, alpha=0.7, color="#dc2626", label="marcado")
axes[0].set_xlabel(f"PC1 ({explained[0]:.0%} de la varianza)")
axes[0].set_ylabel(f"PC2 ({explained[1]:.0%} de la varianza)")
axes[0].set_title("Clientes proyectados en 2D")
axes[0].legend()
weights = pd.Series(components[0], index=dims).sort_values()
axes[1].barh(weights.index, weights.values, color="#2563eb")
axes[1].set_title("¿Qué dimensiones forman PC1?")
fig.tight_layout()
plt.show()

## 3. Búsqueda exacta de vecinos (k-NN)

Normalizamos los vectores y usamos **similitud coseno** (1 = misma dirección). La búsqueda exacta compara la consulta contra **todos** los vectores: costo O(n).

In [ ]:
Xn = X / np.linalg.norm(X, axis=1, keepdims=True)

def knn(query_index, k=10, candidates=None):
    pool = np.arange(len(Xn)) if candidates is None else candidates
    pool = pool[pool != query_index]
    sims = Xn[pool] @ Xn[query_index]
    top = np.argsort(-sims)[:k]
    return pool[top], sims[top]

query = int(np.flatnonzero(flagged)[0])
neighbors, sims = knn(query)
print(f"Consulta: cliente {ids[query]} (marcado)")
display(pd.DataFrame({"customer_id": ids[neighbors], "similitud": sims.round(4), "marcado": flagged[neighbors],
                      "avg_amount": (X[neighbors, -2] * 2000).round(2), "high_ticket_share": X[neighbors, -1].round(2)}))

### ¿Los vecinos sirven para clasificar?

Para 1.000 clientes al azar miramos qué proporción de sus 10 vecinos está marcada (clasificación k-NN de la teoría).

In [ ]:
rng = np.random.default_rng(3)
sample = rng.choice(len(Xn), size=min(1000, len(Xn)), replace=False)
neighbor_rate = np.array([flagged[knn(i)[0]].mean() for i in sample])
rate_if_flagged = neighbor_rate[flagged[sample]].mean()
rate_if_not = neighbor_rate[~flagged[sample]].mean()
accuracy = ((neighbor_rate >= 0.5) == flagged[sample]).mean()

fig, ax = plt.subplots(figsize=(7, 3.5))
labels = ["tasa base", "vecinos de\nno marcados", "vecinos de\nmarcados"]
values = [100 * flagged.mean(), 100 * rate_if_not, 100 * rate_if_flagged]
ax.bar(labels, values, color=["#94a3b8", "#60a5fa", "#dc2626"])
for i, v in enumerate(values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
ax.set_ylabel("% marcados")
ax.set_title("¿Qué tan marcados están los 10 vecinos más parecidos?")
plt.show()
print(f"Marcados entre los vecinos: de clientes marcados {rate_if_flagged:.1%} · de no marcados {rate_if_not:.1%} · tasa base {flagged.mean():.1%}")
print(f"Exactitud del clasificador k-NN (voto de 10 vecinos): {accuracy:.1%} · predecir siempre 'no marcado': {1 - flagged[sample].mean():.1%}")

## 4. Búsqueda aproximada (ANN) con un índice IVF

Con millones de vectores, comparar contra todos es inviable. Un índice **IVF** (*inverted file*) agrupa los vectores en `n_lists` clusters con k-means. Al buscar, sólo revisa los `nprobe` clusters más cercanos a la consulta. Escanea menos, pero puede perder algún vecino verdadero: el trade-off **recall vs. costo**.

Esta es también la forma de **particionar** una base vectorial distribuida: cada cluster puede vivir en un nodo distinto.

In [ ]:
def kmeans(data, k, iterations=15, seed=7):
    rng = np.random.default_rng(seed)
    centroids = data[rng.choice(len(data), size=k, replace=False)].copy()
    for _ in range(iterations):
        assignment = np.argmax(data @ centroids.T, axis=1)
        for c in range(k):
            members = data[assignment == c]
            if len(members):
                centroids[c] = members.mean(axis=0)
        centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
    return centroids, np.argmax(data @ centroids.T, axis=1)

n_lists = int(np.sqrt(len(Xn)))
centroids, assignment = kmeans(Xn, n_lists)

def ann(query_index, k=10, nprobe=4):
    closest = np.argsort(-(centroids @ Xn[query_index]))[:nprobe]
    candidates = np.flatnonzero(np.isin(assignment, closest))
    found, _ = knn(query_index, k, candidates)
    return found, len(candidates)

queries = rng.choice(len(Xn), size=min(200, len(Xn)), replace=False)
exact = {q: set(knn(q)[0]) for q in queries}
curve = []
for nprobe in [1, 2, 4, 8, 16, n_lists]:
    recalls, scanned = [], []
    for q in queries:
        found, n_scanned = ann(q, nprobe=nprobe)
        recalls.append(len(exact[q] & set(found)) / 10)
        scanned.append(n_scanned / len(Xn))
    curve.append({"nprobe": nprobe, "recall@10": np.mean(recalls), "escaneado_%": 100 * np.mean(scanned)})
curve = pd.DataFrame(curve)
display(curve.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve["escaneado_%"], curve["recall@10"], marker="o", color="#7c3aed")
for row in curve.itertuples():
    ax.annotate(f"nprobe={row.nprobe}", (row._3, row._2), textcoords="offset points", xytext=(5, -12), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("% de vectores comparados (escala log)")
ax.set_ylabel("Recall@10")
ax.set_title(f"IVF con {n_lists} clusters: menos trabajo, algo menos de exactitud")
ax.grid(alpha=0.3)
plt.show()

### Insertar un vector nuevo

Llega un cliente nuevo parecido al de la consulta. No hace falta reentrenar el índice: se asigna al centroide más cercano y ya se puede encontrar. Si llegan muchos vectores con otra distribución, los clusters quedan desbalanceados y conviene reconstruir el índice.

In [ ]:
new_vector = X[query] + rng.normal(0, 0.01, size=X.shape[1]).astype("float32")
new_vector = new_vector / np.linalg.norm(new_vector)
new_list = int(np.argmax(centroids @ new_vector))
Xn = np.vstack([Xn, new_vector])
assignment = np.append(assignment, new_list)
found, n_scanned = ann(len(Xn) - 1, nprobe=2)
print(f"Vector nuevo asignado al cluster {new_list}. Su vecino más cercano: cliente {ids[found[0]]} (consulta original: {ids[query]})")
Xn, assignment = Xn[:-1], assignment[:-1]   # deshacemos la inserción para el resto del notebook